# Provincial reporting readiness profile

## Purpose

Profile whether canonical projects can be assigned to official PSGC provinces without changing any persisted table. This notebook is evidence for the pending reporting-grain decision. It does not approve a province model, publish a project-province map, or supersede D-34.


## Grain

The final temporary readiness view has exactly one row per canonical `silver_project` in the selected project run. Candidate generation may fan out, but name and spatial candidates are aggregated back to project grain before they are combined.


## 1. Prove the selected inputs are safe

The profile reads validated Silver projects, official PSGC places, the selected boundary snapshot, and the existing project-region result. It creates temporary views only.


In [0]:
%sql
USE CATALOG `buildabida-capstone`;

CREATE OR REPLACE TEMPORARY VIEW latest_province_profile_boundary_load AS
SELECT * EXCEPT (terminal_rank)
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY table_name
            ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
        ) AS terminal_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'boundaries'
      AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
) AS ranked
WHERE terminal_rank = 1;

CREATE OR REPLACE TEMPORARY VIEW latest_province_profile_boundary_validation AS
SELECT * EXCEPT (validation_rank)
FROM (
    SELECT
        *,
        DENSE_RANK() OVER (ORDER BY run_ts DESC, run_id DESC) AS validation_rank
    FROM `04-validation`.dq_results
    WHERE table_name = 'boundaries'
      AND snapshot_id = (
          SELECT snapshot_id FROM latest_province_profile_boundary_load
      )
) AS ranked
WHERE validation_rank = 1;

CREATE OR REPLACE TEMPORARY VIEW selected_province_profile_project_validation AS
SELECT *
FROM `04-validation`.silver_dq_results
WHERE table_name IN ('silver_project', 'silver_dpwh_project_component')
  AND pipeline_run_id = (
      SELECT MAX(run_id) FROM `02-silver`.silver_project
  );

CREATE OR REPLACE TEMPORARY VIEW selected_province_profile_psgc_validation AS
SELECT *
FROM `04-validation`.silver_dq_results
WHERE table_name = 'silver_psgc_place'
  AND pipeline_run_id = (
      SELECT MAX(run_id)
      FROM `02-silver`.silver_psgc_place
      WHERE place_type = 'PROVINCE'
  );

SELECT ASSERT_TRUE(
    COUNT(*) = 1
    AND COUNT_IF(status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')) = 1
    AND COUNT_IF(snapshot_id IS NOT NULL AND TRIM(snapshot_id) <> '') = 1
    AND COUNT_IF(rows_loaded > 0) = 1,
    'STOP: latest boundary Bronze load is not safe for province profiling'
)
FROM latest_province_profile_boundary_load;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) = 0,
    'STOP: latest boundary validation is missing or unsafe'
)
FROM latest_province_profile_boundary_validation;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) = 0,
    'STOP: selected Silver project validation is missing or unsafe'
)
FROM selected_province_profile_project_validation;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) = 0,
    'STOP: selected Silver PSGC validation is missing or unsafe'
)
FROM selected_province_profile_psgc_validation;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(DISTINCT source_snapshot_id) = 1
    AND COUNT(*) = COUNT(DISTINCT project_key),
    'STOP: silver_project is empty, mixed-run, or not one row per project'
)
FROM `02-silver`.silver_project;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(DISTINCT source_snapshot_id) = 1
    AND COUNT(*) = COUNT(DISTINCT psgc_code),
    'STOP: selected official PSGC province set is empty, mixed-run, or duplicated'
)
FROM `02-silver`.silver_psgc_place
WHERE place_type = 'PROVINCE';

SELECT ASSERT_TRUE(
    COUNT(*) = (SELECT COUNT(*) FROM `02-silver`.silver_project)
    AND COUNT(*) = COUNT(DISTINCT project_key),
    'STOP: current project-region evidence is missing or not one row per project'
)
FROM `02-silver`.silver_project_region_map;


## 2. Prepare canonical project, source-name, PSGC, and boundary evidence

DPWH `location.province` is preserved as source evidence only. Case and whitespace are normalized for exact comparison. The profile neither treats the text as a foreign key nor accepts district-office text. Province boundaries must resolve to the selected official PSGC snapshot and have valid geometry.


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW province_profile_parameters AS
SELECT
    CAST(4.2 AS DOUBLE) AS min_latitude,
    CAST(21.3 AS DOUBLE) AS max_latitude,
    CAST(116.0 AS DOUBLE) AS min_longitude,
    CAST(127.0 AS DOUBLE) AS max_longitude;

CREATE OR REPLACE TEMPORARY VIEW province_profile_projects AS
SELECT
    project.project_key,
    project.contract_id,
    project.reported_budget_pesos,
    project.latitude_parsed AS latitude,
    project.longitude_parsed AS longitude,
    CASE
        WHEN project.coordinate_resolution_status <> 'RESOLVED' THEN 'MISSING_OR_CONFLICTING'
        WHEN project.latitude_parsed IS NULL OR project.longitude_parsed IS NULL THEN 'MISSING_OR_CONFLICTING'
        WHEN project.latitude_parsed NOT BETWEEN parameters.min_latitude AND parameters.max_latitude
          OR project.longitude_parsed NOT BETWEEN parameters.min_longitude AND parameters.max_longitude
            THEN 'OUT_OF_RANGE'
        ELSE 'VALID_PAIR'
    END AS coordinate_status,
    project.source_snapshot_id AS project_source_snapshot_id,
    project.source_ingest_run_id AS project_source_ingest_run_id,
    project.run_id AS project_run_id
FROM `02-silver`.silver_project AS project
CROSS JOIN province_profile_parameters AS parameters;

CREATE OR REPLACE TEMPORARY VIEW province_profile_source_names AS
WITH source_rows AS (
    SELECT
        project.project_key,
        NULLIF(TRIM(GET_JSON_OBJECT(bronze.source_record_json, '$.location.province')), '')
            AS reported_province_raw,
        REGEXP_REPLACE(
            UPPER(NULLIF(TRIM(GET_JSON_OBJECT(
                bronze.source_record_json, '$.location.province'
            )), '')),
            '\\s+',
            ' '
        ) AS reported_province_standardized
    FROM province_profile_projects AS project
    INNER JOIN `01-bronze`.dpwh_projects AS bronze
        ON project.contract_id = bronze.contract_id
       AND project.project_source_snapshot_id = bronze._source_snapshot_id
),
aggregated AS (
    SELECT
        project_key,
        COUNT(DISTINCT reported_province_standardized) AS distinct_source_province_count,
        ARRAY_SORT(COLLECT_SET(reported_province_raw)) AS reported_province_raw_values,
        CASE
            WHEN COUNT(DISTINCT reported_province_standardized) = 1
                THEN MAX(reported_province_standardized)
        END AS reported_province_standardized
    FROM source_rows
    GROUP BY project_key
)
SELECT
    project.project_key,
    COALESCE(source.distinct_source_province_count, 0) AS distinct_source_province_count,
    COALESCE(source.reported_province_raw_values, CAST(ARRAY() AS ARRAY<STRING>))
        AS reported_province_raw_values,
    source.reported_province_standardized
FROM province_profile_projects AS project
LEFT JOIN aggregated AS source
    ON project.project_key = source.project_key;

CREATE OR REPLACE TEMPORARY VIEW province_profile_official_provinces AS
SELECT
    psgc_code AS psgc_province_code,
    place_name_raw AS psgc_province_name,
    place_name_standardized AS psgc_province_name_standardized,
    region_code AS parent_region_code,
    region_name AS parent_region_name,
    psgc_version,
    source_snapshot_id AS psgc_source_snapshot_id,
    source_ingest_run_id AS psgc_source_ingest_run_id,
    run_id AS psgc_run_id
FROM `02-silver`.silver_psgc_place
WHERE place_type = 'PROVINCE';

CREATE OR REPLACE TEMPORARY VIEW province_profile_boundary_inventory AS
SELECT
    boundary.source_feature_id,
    boundary.psgc_code AS boundary_psgc_province_code,
    boundary.source_psgc_version AS boundary_psgc_version,
    boundary.source_boundary_version AS boundary_version,
    boundary._source_snapshot_id AS boundary_source_snapshot_id,
    boundary._ingest_run_id AS boundary_source_ingest_run_id,
    province.psgc_province_code AS resolved_current_province_code,
    TRY_TO_GEOMETRY(REGEXP_REPLACE(
        REGEXP_REPLACE(
            boundary.geometry_json,
            '"(\\[[^"]*\\])"',
            CONCAT(CHAR(36), '1')
        ),
        '"(-?[0-9]+(?:\\.[0-9]+)?(?:[Ee][+-]?[0-9]+)?)"',
        CONCAT(CHAR(36), '1')
    )) AS boundary_geometry
FROM `01-bronze`.boundaries AS boundary
LEFT JOIN province_profile_official_provinces AS province
    ON boundary.psgc_code = province.psgc_province_code
CROSS JOIN latest_province_profile_boundary_load AS latest
WHERE LOWER(TRIM(boundary.administrative_level)) = 'province'
  AND boundary._source_snapshot_id = latest.snapshot_id;

CREATE OR REPLACE TEMPORARY VIEW province_profile_usable_boundaries AS
SELECT *
FROM province_profile_boundary_inventory
WHERE resolved_current_province_code IS NOT NULL
  AND boundary_geometry IS NOT NULL
  AND ST_ISVALID(boundary_geometry);


## 3. Generate exact-name and spatial candidates, then collapse immediately

An exact source-name candidate is profiling evidence, not an approved mapping. Spatial matching uses `ST_COVERS` with longitude-first points. Zero and multiple candidates remain visible. No fuzzy, substring, nearest-place, or candidate-order rule is used.


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW province_profile_name_candidates AS
SELECT
    project.project_key,
    COUNT(DISTINCT province.psgc_province_code) AS name_candidate_count,
    ARRAY_SORT(COLLECT_SET(province.psgc_province_code)) AS name_candidate_codes,
    CASE
        WHEN COUNT(DISTINCT province.psgc_province_code) = 1
            THEN ELEMENT_AT(
                ARRAY_SORT(COLLECT_SET(province.psgc_province_code)),
                1
            )
    END AS exact_name_province_code
FROM province_profile_source_names AS project
INNER JOIN province_profile_official_provinces AS province
    ON project.reported_province_standardized
        = province.psgc_province_name_standardized
WHERE project.distinct_source_province_count = 1
GROUP BY project.project_key;

CREATE OR REPLACE TEMPORARY VIEW province_profile_spatial_candidates AS
SELECT /*+ BROADCAST(boundary) */
    project.project_key,
    COUNT(DISTINCT boundary.resolved_current_province_code) AS spatial_candidate_count,
    COUNT(DISTINCT boundary.source_feature_id) AS boundary_feature_count,
    ARRAY_SORT(COLLECT_SET(boundary.resolved_current_province_code))
        AS spatial_candidate_codes,
    CASE
        WHEN COUNT(DISTINCT boundary.resolved_current_province_code) = 1
            THEN ELEMENT_AT(
                ARRAY_SORT(COLLECT_SET(boundary.resolved_current_province_code)),
                1
            )
    END AS spatial_province_code
FROM province_profile_projects AS project
INNER JOIN province_profile_usable_boundaries AS boundary
    ON ST_COVERS(
        boundary.boundary_geometry,
        ST_POINT(project.longitude, project.latitude, 4326)
    )
WHERE project.coordinate_status = 'VALID_PAIR'
GROUP BY project.project_key;

CREATE OR REPLACE TEMPORARY VIEW project_province_readiness_profile AS
WITH evidence AS (
    SELECT
        project.*,
        source.reported_province_raw_values,
        source.reported_province_standardized,
        source.distinct_source_province_count,
        COALESCE(name.name_candidate_count, 0) AS name_candidate_count,
        COALESCE(name.name_candidate_codes, CAST(ARRAY() AS ARRAY<STRING>))
            AS name_candidate_codes,
        name.exact_name_province_code,
        COALESCE(spatial.spatial_candidate_count, 0) AS spatial_candidate_count,
        COALESCE(spatial.boundary_feature_count, 0) AS boundary_feature_count,
        COALESCE(spatial.spatial_candidate_codes, CAST(ARRAY() AS ARRAY<STRING>))
            AS spatial_candidate_codes,
        spatial.spatial_province_code
    FROM province_profile_projects AS project
    INNER JOIN province_profile_source_names AS source
        ON project.project_key = source.project_key
    LEFT JOIN province_profile_name_candidates AS name
        ON project.project_key = name.project_key
    LEFT JOIN province_profile_spatial_candidates AS spatial
        ON project.project_key = spatial.project_key
)
SELECT
    evidence.*,
    CASE
        WHEN name_candidate_count = 1 AND spatial_candidate_count = 1
         AND exact_name_province_code = spatial_province_code
            THEN 'NAME_COORDINATE_AGREE'
        WHEN name_candidate_count = 1 AND spatial_candidate_count = 1
         AND exact_name_province_code <> spatial_province_code
            THEN 'NAME_COORDINATE_CONFLICT'
        WHEN name_candidate_count = 1 AND spatial_candidate_count = 0
            THEN 'EXACT_NAME_ONLY'
        WHEN name_candidate_count = 0 AND spatial_candidate_count = 1
            THEN 'UNIQUE_COORDINATE_ONLY'
        WHEN name_candidate_count > 1 OR spatial_candidate_count > 1
            THEN 'AMBIGUOUS'
        ELSE 'UNMAPPED'
    END AS evidence_group
FROM evidence;

SELECT ASSERT_TRUE(
    COUNT(*) = (SELECT COUNT(*) FROM `02-silver`.silver_project)
    AND COUNT(*) = COUNT(DISTINCT project_key),
    'STOP: readiness profiling changed the canonical project grain'
)
FROM project_province_readiness_profile;

SELECT ASSERT_TRUE(
    SUM(reported_budget_pesos) <=> (
        SELECT SUM(reported_budget_pesos) FROM `02-silver`.silver_project
    ),
    'STOP: readiness profiling changed the canonical reported-budget total'
)
FROM project_province_readiness_profile;


## 4. Required project, budget, conflict, and special-area results

Run and export each result. Percentages use the complete canonical project and reported-budget totals. A unique candidate is not an accepted production mapping until the team approves a reporting-area contract and mapping precedence.


In [0]:
%sql
WITH totals AS (
    SELECT
        COUNT(*) AS total_projects,
        SUM(reported_budget_pesos) AS total_reported_budget_pesos
    FROM project_province_readiness_profile
)
SELECT
    COUNT(*) AS total_projects,
    COUNT_IF(coordinate_status = 'VALID_PAIR') AS projects_with_valid_coordinates,
    COUNT_IF(coordinate_status <> 'VALID_PAIR') AS projects_without_valid_coordinates,
    COUNT_IF(spatial_candidate_count = 1) AS projects_in_exactly_one_province_polygon,
    COUNT_IF(coordinate_status = 'VALID_PAIR' AND spatial_candidate_count = 0)
        AS valid_coordinate_projects_in_zero_province_polygons,
    COUNT_IF(spatial_candidate_count > 1) AS projects_in_multiple_province_polygons,
    COUNT_IF(name_candidate_count = 1) AS projects_with_exact_official_province_name,
    COUNT_IF(
        distinct_source_province_count = 1 AND name_candidate_count = 0
    ) AS projects_with_nonofficial_source_province_text,
    COUNT_IF(evidence_group = 'NAME_COORDINATE_AGREE')
        AS projects_where_name_and_coordinate_agree,
    COUNT_IF(evidence_group = 'NAME_COORDINATE_CONFLICT')
        AS projects_where_name_and_coordinate_conflict,
    ROUND(
        100.0 * COUNT_IF(spatial_candidate_count = 1) / NULLIF(MAX(totals.total_projects), 0),
        2
    ) AS unique_spatial_project_coverage_pct,
    SUM(reported_budget_pesos) AS total_reported_budget_pesos,
    SUM(CASE WHEN spatial_candidate_count = 1 THEN reported_budget_pesos END)
        AS unique_spatial_reported_budget_pesos,
    ROUND(
        100.0 * SUM(CASE WHEN spatial_candidate_count = 1 THEN reported_budget_pesos END)
        / NULLIF(MAX(totals.total_reported_budget_pesos), 0),
        2
    ) AS unique_spatial_budget_coverage_pct
FROM project_province_readiness_profile
CROSS JOIN totals;

SELECT
    evidence_group,
    COUNT(*) AS projects,
    COUNT_IF(reported_budget_pesos IS NOT NULL) AS projects_with_reported_budget,
    SUM(reported_budget_pesos) AS reported_budget_pesos,
    COUNT_IF(coordinate_status = 'VALID_PAIR') AS projects_with_valid_coordinates,
    COUNT_IF(distinct_source_province_count = 0) AS projects_without_source_province_text,
    COUNT_IF(distinct_source_province_count > 1) AS projects_with_conflicting_source_province_text
FROM project_province_readiness_profile
GROUP BY evidence_group
ORDER BY projects DESC, evidence_group;

SELECT
    CASE
        WHEN SUBSTRING(region.psgc_region_code, 1, 2) = '13' THEN 'NCR'
        WHEN SUBSTRING(region.psgc_region_code, 1, 2) = '19' THEN 'BARMM'
        ELSE 'OTHER_REGIONS'
    END AS region_group,
    COUNT(*) AS projects,
    SUM(profile.reported_budget_pesos) AS reported_budget_pesos,
    COUNT_IF(profile.spatial_candidate_count = 1) AS unique_spatial_province_candidates,
    COUNT_IF(profile.spatial_candidate_count = 0) AS zero_spatial_province_candidates,
    COUNT_IF(profile.spatial_candidate_count > 1) AS multiple_spatial_province_candidates,
    COUNT_IF(profile.name_candidate_count = 1) AS exact_official_province_names,
    COUNT_IF(profile.evidence_group = 'NAME_COORDINATE_CONFLICT') AS conflicts
FROM project_province_readiness_profile AS profile
INNER JOIN `02-silver`.silver_project_region_map AS region
    ON profile.project_key = region.project_key
GROUP BY region_group
ORDER BY region_group;


## 5. Boundary-version and independent-city contract gaps

The current `silver_psgc_place` contract identifies cities but does not distinguish highly urbanized from independent component cities. It also gives some cities no province parent. Report the combined no-province city count; do not relabel it as either HUC or ICC without an approved PSGC classification field.


In [0]:
%sql
SELECT
    'BOUNDARY_PROVINCE_ROWS' AS finding,
    COUNT(*) AS affected_rows,
    COUNT(DISTINCT boundary_psgc_version) AS distinct_boundary_psgc_versions,
    COUNT(DISTINCT boundary_version) AS distinct_boundary_versions
FROM province_profile_boundary_inventory
UNION ALL
SELECT
    'BOUNDARY_CODES_NOT_IN_CURRENT_PSGC' AS finding,
    COUNT(*) AS affected_rows,
    COUNT(DISTINCT boundary_psgc_version) AS distinct_boundary_psgc_versions,
    COUNT(DISTINCT boundary_version) AS distinct_boundary_versions
FROM province_profile_boundary_inventory
WHERE resolved_current_province_code IS NULL
UNION ALL
SELECT
    'CURRENT_PSGC_PROVINCES_WITHOUT_BOUNDARY' AS finding,
    COUNT(*) AS affected_rows,
    CAST(NULL AS BIGINT) AS distinct_boundary_psgc_versions,
    CAST(NULL AS BIGINT) AS distinct_boundary_versions
FROM province_profile_official_provinces AS province
LEFT ANTI JOIN province_profile_boundary_inventory AS boundary
    ON province.psgc_province_code = boundary.boundary_psgc_province_code
UNION ALL
SELECT
    'INVALID_OR_UNPARSEABLE_PROVINCE_BOUNDARIES' AS finding,
    COUNT(*) AS affected_rows,
    COUNT(DISTINCT boundary_psgc_version) AS distinct_boundary_psgc_versions,
    COUNT(DISTINCT boundary_version) AS distinct_boundary_versions
FROM province_profile_boundary_inventory
WHERE boundary_geometry IS NULL OR NOT ST_ISVALID(boundary_geometry);

SELECT
    COUNT_IF(place_type = 'CITY') AS official_city_rows,
    COUNT_IF(place_type = 'CITY' AND province_code IS NULL)
        AS city_rows_without_official_province_parent,
    'NOT_AVAILABLE_IN_CURRENT_SILVER_CONTRACT' AS huc_vs_icc_classification_status
FROM `02-silver`.silver_psgc_place;

SELECT
    reported_province_standardized,
    COUNT(*) AS projects,
    SUM(reported_budget_pesos) AS reported_budget_pesos,
    COUNT_IF(coordinate_status = 'VALID_PAIR') AS projects_with_valid_coordinates,
    COUNT_IF(spatial_candidate_count = 1) AS projects_with_unique_spatial_candidate
FROM project_province_readiness_profile
WHERE distinct_source_province_count = 1
  AND name_candidate_count = 0
GROUP BY reported_province_standardized
ORDER BY projects DESC, reported_province_standardized;


# Summary

- **Built:** no persisted table. The notebook creates session-scoped views and emits readiness evidence only.
- **Protected:** the final profile asserts the canonical project count and reported-budget total are unchanged.
- **Not approved:** source-name and coordinate candidates are not production mappings. D-34 remains in force.
- **Next:** export the result sets, review unexpected source values and boundary-version gaps, approve either strict provinces or governed province reporting areas, then implement the selected Silver, Gold, Analytics, and validation contracts.
